In [1]:
!pip install pydriller pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.0 MB/s eta 0:00:00


In [2]:
from pydriller import Repository
import re

In [3]:
!git clone https://github.com/apache/camel.git

Cloning into 'camel'...
remote: Enumerating objects: 4082342, done.
remote: Counting objects: 100% (3682/3682), done.
remote: Compressing objects: 100% (333/333), done.
remote: Total 4082342 (delta 3510), reused 3355 (delta 3342), pack-reused 4078660 (from 4)
Receiving objects: 100% (4082342/4082342), 1.19 GiB | 17.58 MiB/s, done.
Resolving deltas: 100% (1404178/1404178), done.
Updating files: 100% (37998/37998), done.


In [7]:
REPO_PATH = "/content/camel" #cloned it in google colab

ISSUE_IDS = [
    "CAMEL-180",
    "CAMEL-321",
    "CAMEL-1818",
    "CAMEL-3214",
    "CAMEL-18065"
]

In [8]:
def is_issue_commit(message, issue_ids):
    """Check if commit message contains any of the issue IDs"""
    return any(re.search(issue_id, message) for issue_id in issue_ids)


def main():
    unique_commit_hashes = set()
    selected_commits = []

    print("Scanning the repo")

    # Traverse all the commits
    for commit in Repository(REPO_PATH).traverse_commits():

        # Skip merge commits
        if len(commit.parents) > 1:
            continue

        # Check if commit matches any issue ID
        if is_issue_commit(commit.msg, ISSUE_IDS):

            if commit.hash not in unique_commit_hashes:
                unique_commit_hashes.add(commit.hash)
                selected_commits.append(commit)

    total_commits = len(selected_commits)

    if total_commits == 0:
        print("No matching commits found.")
        return


    unique_files = set()
    dmm_scores = []

    for commit in selected_commits:

        # Collect changed files
        for file in commit.modified_files:
            if file.change_type.name in ["ADD", "MODIFY", "DELETE"]:

                if file.new_path:
                    unique_files.add(file.new_path)
                elif file.old_path:
                    unique_files.add(file.old_path)

        # Calculate DMM score
        if (
            commit.dmm_unit_size is not None and
            commit.dmm_unit_complexity is not None and
            commit.dmm_unit_interfacing is not None
        ):
            dmm = (
                commit.dmm_unit_size +
                commit.dmm_unit_complexity +
                commit.dmm_unit_interfacing
            ) / 3

            dmm_scores.append(dmm)


    avg_files_changed = len(unique_files) / total_commits
    avg_dmm = sum(dmm_scores) / len(dmm_scores) if dmm_scores else 0

    print("\n=== RESULTS ===")
    print(f"Total commits analyzed: {total_commits}")
    print(f"Average number of files changed: {avg_files_changed:.2f}")
    print(f"Average DMM metrics: {avg_dmm:.2f}")



if __name__ == "__main__":
    main()


Scanning the repo

=== RESULTS ===
Total commits analyzed: 169
Average number of files changed: 3.75
Average DMM metrics: 0.60
